# 模块二实验：分布式训练对比实验

**目标**：
1. 对比单卡 FP32 / BF16 混合精度 / 梯度检查点 的显存与速度
2. 理解 ZeRO/FSDP 如何通过分片减少显存占用
3. 可视化各配置的吞吐量与显存权衡

**环境说明**：
- 有 GPU（≥8GB 显存）：可运行完整实验
- Apple Silicon MPS：部分实验可运行（FSDP 不支持 MPS）
- 仅 CPU：规模缩小，所有实验可运行（速度较慢）

```bash
pip install torch matplotlib numpy
```

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.checkpoint import checkpoint
import numpy as np
import matplotlib.pyplot as plt
import time, gc

# 环境检测
if torch.cuda.is_available():
    device = torch.device('cuda')
    total_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'显存: {total_mem_gb:.1f} GB')
    HAS_GPU = True
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
    print('Apple Silicon MPS（部分实验可运行）')
    HAS_GPU = False
else:
    device = torch.device('cpu')
    print('CPU 模式（使用缩小配置）')
    HAS_GPU = False

# 根据硬件选配置
if HAS_GPU:
    CONFIG = dict(n_layers=12, d_model=768, n_heads=12, d_ff=3072,
                  seq_len=512, batch_size=8)   # GPT-2 Small
else:
    CONFIG = dict(n_layers=4, d_model=256, n_heads=4, d_ff=1024,
                  seq_len=128, batch_size=4)    # 缩小版，CPU可跑

print(f'\n实验配置: {CONFIG}')
param_count = CONFIG['n_layers'] * (
    4 * CONFIG['d_model']**2 +     # QKV + Output proj
    2 * CONFIG['d_model'] * CONFIG['d_ff']  # FFN
) / 1e6
print(f'估算参数量: ~{param_count:.0f}M')

---
## Part 1：定义模型

In [ ]:
class GPTBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.attn  = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.ffn   = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model)
        )
        self.ln1   = nn.LayerNorm(d_model)
        self.ln2   = nn.LayerNorm(d_model)

    def forward(self, x):
        a, _ = self.attn(x, x, x, need_weights=False)
        x    = self.ln1(x + a)
        x    = self.ln2(x + self.ffn(x))
        return x

class GPT(nn.Module):
    def __init__(self, cfg, use_ckpt=False):
        super().__init__()
        self.use_ckpt = use_ckpt
        self.embed    = nn.Embedding(50257, cfg['d_model'])
        self.blocks   = nn.ModuleList([
            GPTBlock(cfg['d_model'], cfg['n_heads'], cfg['d_ff'])
            for _ in range(cfg['n_layers'])
        ])
        self.head     = nn.Linear(cfg['d_model'], 50257, bias=False)

    def forward(self, idx):
        x = self.embed(idx)
        for block in self.blocks:
            if self.use_ckpt and self.training:
                x = checkpoint(block, x, use_reentrant=False)
            else:
                x = block(x)
        return self.head(x)

def make_model(use_ckpt=False):
    return GPT(CONFIG, use_ckpt=use_ckpt).to(device)

def make_batch():
    return torch.randint(0, 50257,
        (CONFIG['batch_size'], CONFIG['seq_len']), device=device)

m = make_model()
real_params = sum(p.numel() for p in m.parameters()) / 1e6
print(f'实际参数量: {real_params:.1f}M')
del m; gc.collect()

---
## Part 2：基准测试工具函数

In [ ]:
def get_mem_gb():
    if torch.cuda.is_available():
        return torch.cuda.max_memory_allocated() / 1e9
    return 0.0  # CPU/MPS 无法精确测量

def reset_mem():
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.empty_cache()
    gc.collect()

def benchmark(label, model, optimizer, use_amp=False, n_steps=20, warmup=5):
    """测量训练吞吐量（tokens/s）和显存峰值"""
    model.train()
    scaler = torch.amp.GradScaler('cuda') if (use_amp and HAS_GPU) else None
    criterion = nn.CrossEntropyLoss()

    reset_mem()
    times = []

    for step in range(n_steps + warmup):
        x = make_batch()
        y = torch.roll(x, -1, dims=1)  # next-token prediction

        t0 = time.perf_counter()
        optimizer.zero_grad()

        if use_amp and HAS_GPU:
            with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                logits = model(x)
                loss   = criterion(logits.view(-1, 50257), y.view(-1))
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(x)
            loss   = criterion(logits.view(-1, 50257), y.view(-1))
            loss.backward()
            optimizer.step()

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        t1 = time.perf_counter()
        if step >= warmup:
            times.append(t1 - t0)

    tokens_per_step = CONFIG['batch_size'] * CONFIG['seq_len']
    avg_time   = np.mean(times)
    throughput = tokens_per_step / avg_time
    peak_mem   = get_mem_gb()

    print(f'{label:<35} | {throughput:>9,.0f} tok/s | 显存峰值 {peak_mem:.2f} GB | {avg_time*1000:.1f} ms/step')
    return {'label': label, 'throughput': throughput, 'peak_mem': peak_mem, 'step_time_ms': avg_time*1000}

---
## Part 3：实验对比

In [ ]:
results = []
print(f'{"配置":<35} | {"吞吐量":>9} | {"显存":>12} | {"每步耗时"}')
print('-' * 75)

# ── 实验1：FP32 基准 ──
reset_mem()
m1 = make_model(use_ckpt=False)
opt1 = optim.AdamW(m1.parameters(), lr=3e-4)
results.append(benchmark('①  FP32 基准', m1, opt1, use_amp=False))
del m1, opt1; reset_mem()

# ── 实验2：BF16 混合精度 ──
reset_mem()
m2 = make_model(use_ckpt=False)
opt2 = optim.AdamW(m2.parameters(), lr=3e-4)
results.append(benchmark('②  BF16 混合精度（AMP）', m2, opt2, use_amp=True))
del m2, opt2; reset_mem()

# ── 实验3：FP32 + 梯度检查点 ──
reset_mem()
m3 = make_model(use_ckpt=True)
opt3 = optim.AdamW(m3.parameters(), lr=3e-4)
results.append(benchmark('③  FP32 + 梯度检查点', m3, opt3, use_amp=False))
del m3, opt3; reset_mem()

# ── 实验4：BF16 + 梯度检查点 ──
reset_mem()
m4 = make_model(use_ckpt=True)
opt4 = optim.AdamW(m4.parameters(), lr=3e-4)
results.append(benchmark('④  BF16 + 梯度检查点', m4, opt4, use_amp=True))
del m4, opt4; reset_mem()

---
## Part 4：ZeRO/FSDP 显存分析（概念演示）

In [ ]:
# ZeRO 三个阶段的显存节省量（理论分析）
def zero_memory_analysis(param_gb, n_gpus=8):
    """分析 ZeRO-1/2/3 在 n_gpus 张GPU上的显存分布"""
    # 一个参数的内存占用
    # FP16 权重: P bytes
    # FP32 master 权重: 2P
    # FP32 梯度: 2P
    # FP32 优化器状态 (Adam: m+v): 4P+4P = 8P
    # 总计: 16P bytes（以 FP16 参数为1单位）
    P = param_gb
    baseline = 16 * P  # 每张GPU都有全量

    zero1 = P + 2*P + 2*P + (8*P / n_gpus)  # 优化器状态分片
    zero2 = P + (2*P / n_gpus) + (2*P / n_gpus) + (8*P / n_gpus)  # +梯度分片
    zero3 = (P / n_gpus) + (2*P / n_gpus) + (2*P / n_gpus) + (8*P / n_gpus)  # +权重分片

    print(f'模型参数: {P:.1f} GB (FP16)')
    print(f'GPU 数量: {n_gpus}')
    print()
    print(f'无分片 (DDP):  {baseline:.1f} GB/GPU')
    print(f'ZeRO-1:        {zero1:.1f} GB/GPU  ({baseline/zero1:.1f}x 节省)')
    print(f'ZeRO-2:        {zero2:.1f} GB/GPU  ({baseline/zero2:.1f}x 节省)')
    print(f'ZeRO-3:        {zero3:.1f} GB/GPU  ({baseline/zero3:.1f}x 节省)')
    return baseline, zero1, zero2, zero3

print('=== LLaMA-2-7B (7B params, FP16 ≈ 14GB) 在 8×GPU 集群的显存分布 ===')
vals = zero_memory_analysis(param_gb=14.0, n_gpus=8)

print()
print('=== LLaMA-2-70B (70B params, FP16 ≈ 140GB) 在 8×GPU 集群的显存分布 ===')
vals70 = zero_memory_analysis(param_gb=140.0, n_gpus=8)

In [ ]:
# 可视化 ZeRO 分析
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('ZeRO 三阶段显存节省分析', fontsize=13, fontweight='bold')

for ax, model_name, param_gb in [
    (axes[0], 'LLaMA-2-7B (14GB FP16)', 14.0),
    (axes[1], 'LLaMA-2-70B (140GB FP16)', 140.0)
]:
    n_gpus_list = [1, 2, 4, 8, 16, 32]
    P = param_gb
    baseline_list = [16*P] * len(n_gpus_list)
    z1_list = [P + 2*P + 2*P + (8*P/g) for g in n_gpus_list]
    z2_list = [P + (4*P/g) + (8*P/g) for g in n_gpus_list]
    z3_list = [(P/g) + (4*P/g) + (8*P/g) for g in n_gpus_list]

    ax.plot(n_gpus_list, baseline_list, 'k--o', label='DDP（无分片）', linewidth=2)
    ax.plot(n_gpus_list, z1_list, 'b-s', label='ZeRO-1（优化器分片）', linewidth=2)
    ax.plot(n_gpus_list, z2_list, 'g-^', label='ZeRO-2（+梯度分片）', linewidth=2)
    ax.plot(n_gpus_list, z3_list, 'r-v', label='ZeRO-3（+权重分片）', linewidth=2)
    ax.axhline(y=80, color='orange', linestyle=':', label='A100 80GB', linewidth=1.5)

    ax.set_xlabel('GPU 数量', fontsize=11)
    ax.set_ylabel('每GPU显存 (GB)', fontsize=11)
    ax.set_title(model_name, fontsize=11)
    ax.set_xticks(n_gpus_list)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('zero_memory_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Part 5：结果可视化

In [ ]:
if results:
    labels      = [r['label'] for r in results]
    throughputs = [r['throughput'] for r in results]
    peak_mems   = [r['peak_mem'] for r in results]

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle('各训练配置性能对比', fontsize=13, fontweight='bold')
    colors = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6']

    # 吞吐量
    bars = axes[0].bar(range(len(labels)), throughputs, color=colors, alpha=0.85, edgecolor='black')
    for bar, v in zip(bars, throughputs):
        axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+50,
                     f'{v:,.0f}', ha='center', fontsize=9)
    axes[0].set_xticks(range(len(labels)))
    axes[0].set_xticklabels([l.split()[0] for l in labels], fontsize=10)
    axes[0].set_ylabel('吞吐量 (tokens/s)')
    axes[0].set_title('① 训练吞吐量（越高越好）')
    axes[0].grid(True, alpha=0.3, axis='y')

    # 显存
    if any(m > 0 for m in peak_mems):
        bars2 = axes[1].bar(range(len(labels)), peak_mems, color=colors, alpha=0.85, edgecolor='black')
        for bar, v in zip(bars2, peak_mems):
            axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
                         f'{v:.2f}GB', ha='center', fontsize=9)
        axes[1].set_xticks(range(len(labels)))
        axes[1].set_xticklabels([l.split()[0] for l in labels], fontsize=10)
        axes[1].set_ylabel('显存峰值 (GB)')
        axes[1].set_title('② 显存峰值（越低越好）')
        axes[1].grid(True, alpha=0.3, axis='y')
    else:
        axes[1].text(0.5, 0.5, 'CPU/MPS\n无法测量显存', ha='center', va='center',
                     transform=axes[1].transAxes, fontsize=12)
        axes[1].set_title('② 显存峰值')

    # 吞吐量 vs 显存散点图
    if any(m > 0 for m in peak_mems):
        for i, (t, m, lab) in enumerate(zip(throughputs, peak_mems, labels)):
            axes[2].scatter(m, t, s=200, color=colors[i], zorder=5)
            axes[2].annotate(lab.split()[0], (m, t),
                             xytext=(5, 5), textcoords='offset points', fontsize=9)
        axes[2].set_xlabel('显存峰值 (GB)')
        axes[2].set_ylabel('吞吐量 (tokens/s)')
        axes[2].set_title('③ 吞吐量 vs 显存权衡\n（右上角最优）')
        axes[2].grid(True, alpha=0.3)
    else:
        axes[2].text(0.5, 0.5, '需要 GPU\n才能绘制', ha='center', va='center',
                     transform=axes[2].transAxes, fontsize=12)

    plt.tight_layout()
    plt.savefig('training_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

    # 打印倍数对比
    base = throughputs[0]
    print('\n=== 相对基准（FP32）的加速比 ===')
    for r in results:
        print(f"{r['label']:<35} {r['throughput']/base:.2f}x")

---
## 思考题

**Q1**：从 Part 3 的实验结果来看，BF16 混合精度相比 FP32 基准快了多少倍？显存节省了多少？这个结果符合你的预期吗？

*(你的答案)*：

---

**Q2**：梯度检查点开启后，显存减少了多少？但吞吐量下降了多少？这说明梯度检查点在什么情况下最有价值（提示：想想 GPU 显存恰好装不下模型的场景）？

*(你的答案)*：

---

**Q3**：从 Part 4 的 ZeRO 分析来看，LLaMA-2-70B 要在 8 张 A100（80GB）上运行，至少需要 ZeRO 几阶段？（不计算激活值显存）

*(你的答案)*：

---

**Q4**：Pipeline Parallelism 的气泡率公式为 `(p-1)/(m+p-1)`，其中 p=流水线阶段数，m=micro-batch数。当 p=4，m=1 时气泡率是多少？m 增大到 16 时呢？这说明 Pipeline Parallelism 适合什么样的 batch size 配置？

*(你的答案)*：